# Record a CARLA sequence on Kaggle

Runs `carla_scripts/record_sequence.py` against a headless CARLA 0.9.15 server and packs the recording into one
`.tar` under Output, in the same layout as the local recordings (`rgb/`, `instance/`, `depth/`,
`frames.parquet`, `actors.parquet`, `meta.json`).

Kaggle gives notebooks compute-only GPU access, so CARLA can't render on the GPU here; it renders through
lavapipe (Mesa's CPU Vulkan) instead. That is slow (~10 s per frame with 3 cameras), but the recorder runs CARLA
in synchronous mode, which waits for every frame, so the data is the same as a fast run.

**One-time setup:** upload `perception-stack-src.zip` as a private Kaggle Dataset, then attach it here with
*Add Input* (right panel).

**Each recording:**
1. Session options: Accelerator **None** (CPU sessions have no weekly quota), Internet **On**.
2. Edit the settings cell below.
3. **Save Version -> Save & Run All (Commit)**. It runs in the background (up to 12 h); the tab can be closed.
4. When it finishes, open the version, go to *Output* and download `<name>.tar` (and `<name>_log.json`).

Time: ~15 min setup (mostly the CARLA download), under a minute to start CARLA, then ~5-10 s per frame
(warmup included): 600 frames ~ 1-2 h.

In [ ]:
# ---- settings ----
TOWN = "Town05"  # works: Town05, Town10HD_Opt; crashes on CPU rendering: Town03; untested: Town01, 02, 04
WEATHER = "ClearNoon"  # carla.WeatherParameters preset, e.g. CloudyNoon, WetNoon, HardRainNoon, ClearSunset,
#                        WetCloudySunset, ClearNight, WetNight
SEED = 2  # traffic and pedestrian layout
FRAMES = 600  # recorded frames at 20 per simulated second
WARMUP = 40  # ticks before recording, to get traffic moving
VEHICLES = 60
WALKERS = 80
SPAWN_RADIUS = (
    150  # metres around the ego; 0 = anywhere (Town05 is large: random spawns leave the road empty)
)
NAME = f"{TOWN.lower().replace('_opt', '')}_{WEATHER.lower()}_s{SEED}"
print("recording name:", NAME)

In [ ]:
import glob
import json
import os
import socket
import subprocess
import threading
import time


def sh(cmd, check=True, quiet=False):
    """Run a bash command, stream its output, return (returncode, last output lines)."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:  # Stop button: don't leave the command running in the background
        proc.kill()
        raise
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}\n{''.join(lines[-30:])}")
    return proc.returncode, "".join(lines[-400:])


LOG = {"name": NAME, "town": TOWN, "weather": WEATHER, "seed": SEED, "frames": FRAMES}
WORK = "/tmp"
CARLA_DIR = f"{WORK}/carla"
REC_DIR = f"{WORK}/rec/{NAME}"
OUT = "/kaggle/working"
sh("free -g; echo cpus: $(nproc); grep PRETTY_NAME /etc/os-release; df -h /tmp | tail -1")

found = glob.glob("/kaggle/input/**/carla_scripts/record_sequence.py", recursive=True)
assert found, "repo not found: attach the perception-stack-src dataset with 'Add Input'"
REPO = os.path.dirname(os.path.dirname(found[0]))
print("repo:", REPO)
assert not os.path.exists(REC_DIR), f"{REC_DIR} exists"

## System packages and Vulkan

In [ ]:
apt = (
    "DEBIAN_FRONTEND=noninteractive timeout 600 apt-get install -y -qq --no-install-recommends "
    "-o DPkg::Lock::Timeout=120"
)
sh("timeout 300 apt-get update -qq", check=False, quiet=True)
sh(f"{apt} libvulkan1 vulkan-tools mesa-vulkan-drivers xdg-user-dirs", quiet=True)
if sh(f"{apt} libomp5", check=False, quiet=True)[0] != 0:  # Ubuntu 24.04: libomp5-<llvm version>
    sh(f"{apt} libomp5-18", quiet=True)
for pkg in ("libsdl2-2.0-0", "libjpeg8", "libpng16-16t64", "fontconfig", "libtiff6", "libx11-6", "libxext6"):
    sh(f"{apt} {pkg}", check=False, quiet=True)

# GPU Vulkan if the container allows it (Kaggle doesn't), otherwise lavapipe on the CPU.
ICD, LOG["renderer"] = None, None
nvidia_glx = "/usr/local/nvidia/lib64/libGLX_nvidia.so.0"
if os.path.exists(nvidia_glx):
    ICD = "/etc/vulkan/icd.d/nvidia_icd.json"
    os.makedirs(os.path.dirname(ICD), exist_ok=True)
    with open(ICD, "w") as f:
        json.dump(
            {"file_format_version": "1.0.0", "ICD": {"library_path": nvidia_glx, "api_version": "1.3.0"}}, f
        )
    _, out = sh(
        f"VK_ICD_FILENAMES={ICD} vulkaninfo --summary 2>&1 | grep -E 'deviceName' || true", check=False
    )
    LOG["renderer"] = "nvidia" if "NVIDIA" in out.upper() or "Tesla" in out else None
if LOG["renderer"] is None:
    ICD = glob.glob("/usr/share/vulkan/icd.d/lvp_icd*.json")[0]
    _, out = sh(f"VK_ICD_FILENAMES={ICD} vulkaninfo --summary 2>&1 | grep -E 'deviceName|ERROR'", check=False)
    assert "llvmpipe" in out, "no working Vulkan driver"
    LOG["renderer"] = "lavapipe (CPU)"
print("renderer:", LOG["renderer"])

## CARLA server and Python 3.10 client

In [ ]:
CARLA_URLS = [
    "https://carla-releases.s3.us-east-005.backblazeb2.com/Linux/CARLA_0.9.15.tar.gz",
    "https://carla-releases.s3.eu-west-3.amazonaws.com/Linux/CARLA_0.9.15.tar.gz",
]
os.makedirs(CARLA_DIR, exist_ok=True)
if not os.path.exists(f"{CARLA_DIR}/CarlaUE4.sh"):
    for url in CARLA_URLS:
        start = time.time()
        if sh(f"curl -fsSL --retry 3 {url} | tar -xz -C {CARLA_DIR}", check=False)[0] == 0:
            LOG["carla_download_min"] = round((time.time() - start) / 60, 1)
            break
        print("failed:", url)
    else:
        raise RuntimeError("could not download CARLA")

# The carla 0.9.15 wheel stops at Python 3.10; Kaggle runs a newer Python, so the client gets its own env.
PY = f"{WORK}/py310/bin/python"
sh("pip install -q uv", quiet=True)
sh(f"uv python install 3.10 && uv venv -q --allow-existing -p 3.10 {WORK}/py310")
sh(f"uv pip install -q -p {PY} carla==0.9.15 'numpy<2' 'opencv-python-headless<4.11' pandas pyarrow")
check = "import carla, perception.io.carla_recording; print('client OK')"
sh(f'cd {REPO} && PYTHONPATH={REPO}/src {PY} -c "{check}"')

In [ ]:
# Start CARLA directly in the target map: switching maps at runtime crashed (segfault) on lavapipe, and the
# map argument on the command line was ignored, so set the default map in the engine config instead.
sh("pkill -f CarlaUE4 || true; sleep 5", check=False, quiet=True)
ini = f"{CARLA_DIR}/CarlaUE4/Config/DefaultEngine.ini"
map_ref = f"/Game/Carla/Maps/{TOWN}.{TOWN}"
sh(f"sed -i -E 's#^(GameDefaultMap|ServerDefaultMap)=.*#\\1={map_ref}#' {ini} && grep -E 'DefaultMap=' {ini}")
# CARLA refuses to run as root. Detached with no pipes back to the kernel, or the cell would never finish.
sh("id carla >/dev/null 2>&1 || useradd -m carla")
sh(f"chown -R carla:carla {CARLA_DIR}")
SERVER_LOG = f"{WORK}/carla_server.log"
start_cmd = (
    f"cd {CARLA_DIR} && VK_ICD_FILENAMES={ICD} exec ./CarlaUE4.sh -RenderOffScreen "
    f"-nosound -quality-level=Low -carla-rpc-port=2000 > {SERVER_LOG} 2>&1"
)
subprocess.Popen(
    ["su", "carla", "-s", "/bin/bash", "-c", start_cmd],
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)


def port_open(port=2000):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", port)) == 0


def server_alive():
    return subprocess.run(["pgrep", "-f", "CarlaUE4-Linux"], capture_output=True).returncode == 0


start = time.time()
while time.time() - start < 1800 and not port_open() and (server_alive() or time.time() - start < 60):
    time.sleep(10)
LOG["server_start_min"] = round((time.time() - start) / 60, 1)
if not port_open():
    sh(f"tail -n 40 {SERVER_LOG}", check=False)
    raise RuntimeError("CARLA server did not start")

with open(f"{WORK}/ping.py", "w") as f:
    f.write(
        "import carla\nc = carla.Client('localhost', 2000)\nc.set_timeout(900)\n"
        "print('server', c.get_server_version(), 'map', c.get_world().get_map().name)\n"
    )
_, out = sh(f"{PY} {WORK}/ping.py")
assert TOWN in out, f"server is not in {TOWN}"

## Record

In [ ]:
# Memory and progress log every 30 s, kept next to the recording (CPU rendering keeps "VRAM" in RAM).
samples = []


def monitor():
    while not done.is_set():
        info = dict(line.split()[:2] for line in open("/proc/meminfo"))
        used_gb = (int(info["MemTotal:"]) - int(info["MemAvailable:"])) / 1e6
        frames = len(glob.glob(f"{REC_DIR}/rgb/*.jpg"))
        samples.append(
            {"t_min": round((time.time() - t0) / 60, 1), "ram_used_gb": round(used_gb, 1), "frames": frames}
        )
        done.wait(30)


done, t0 = threading.Event(), time.time()
threading.Thread(target=monitor, daemon=True).start()
os.makedirs(os.path.dirname(REC_DIR), exist_ok=True)
rc, _ = sh(
    f"cd {REPO} && PYTHONPATH={REPO}/src {PY} -u carla_scripts/record_sequence.py --town {TOWN} "
    f"--weather {WEATHER} --seed {SEED} --frames {FRAMES} --warmup {WARMUP} --vehicles {VEHICLES} "
    f"--walkers {WALKERS} --spawn-radius {SPAWN_RADIUS} --timeout 900 --out {REC_DIR}",
    check=False,
)
done.set()
LOG["record_min"] = round((time.time() - t0) / 60, 1)
LOG["peak_ram_used_gb"] = max((s["ram_used_gb"] for s in samples), default=None)
LOG["server_alive_after"] = server_alive()
LOG["ok"] = rc == 0 and os.path.exists(f"{REC_DIR}/meta.json")
if not LOG["ok"]:
    sh(f"tail -n 40 {SERVER_LOG}", check=False)

## Package

In [ ]:
LOG["memory"] = samples
if LOG["ok"]:
    LOG["meta"] = json.load(open(f"{REC_DIR}/meta.json"))
    sh(f"tar -cf {OUT}/{NAME}.tar -C {os.path.dirname(REC_DIR)} {NAME} && ls -lh {OUT}/{NAME}.tar")
with open(f"{OUT}/{NAME}_log.json", "w") as f:
    json.dump(LOG, f, indent=2)
sh("pkill -f CarlaUE4 || true", check=False, quiet=True)
print(f"\n{'OK' if LOG['ok'] else 'FAILED'}: {NAME}\n")
print(json.dumps({k: v for k, v in LOG.items() if k != "memory"}, indent=2))